# Experiment 2 — Superposition vs. Parse Failure Discrimination

**Goal**: For the highest-entropy tokens identified in E1, determine whether the model is holding
multiple parses in genuine *superposition* or simply failing to parse (opaque tokens).

**Method**: Run each high-entropy passage through all 6 lenses. Capture residual streams at the
mid-model layer. Project via PCA and examine geometry:
- **Superposed**: token positions fall *between* lens clusters — the model holds all frames.
- **Collapsed**: token positions cluster tightly with one lens, far from others.
- **Opaque**: token positions in low-density region far from all lens clusters.

**Success criterion**: The superposed/collapsed/opaque distribution is non-uniform (χ² p<0.05),
and at least 10% of high-entropy tokens are classified as genuinely superposed.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
from pathlib import Path
from sklearn.decomposition import PCA
from scipy.stats import chi2_contingency

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
USE_REAL_MODEL = False   # Set True if GPU + Llama weights available
E1_RESULTS_PATH = Path('e1_results.json')  # Output from E1

# High-entropy passages from E1 (fallback if E1 not run)
FALLBACK_PASSAGES = [
    {"passage": "riverrun, past Eve and Adam's, from swerve of shore to bend of bay",
     "page": 3, "line": 1},
    {"passage": "commodius vicus of recirculation back to Howth Castle and Environs.",
     "page": 3, "line": 2},
    {"passage": "Sir Tristram, violer d'amores, fr'over the short sea, had passencore",
     "page": 3, "line": 4},
    {"passage": "rearrived from North Armorica on this side the scraggy isthmus of Europe",
     "page": 3, "line": 5},
    {"passage": "wielderfight his penisolate war: nor had topsawyer's rocks by the stream",
     "page": 3, "line": 6},
]

LENS_NAMES = ['viconian', 'kabbalistic', 'freudian', 'irish_mythology', 'norse', 'brunian']
LENS_COLORS = px.colors.qualitative.Plotly[:6]

In [ ]:
# ── Load or synthesise residual stream data ─────────────────────────────────
if USE_REAL_MODEL:
    import torch
    from engine.model.wake_model import WakeModel
    from engine.multipass.runner import MultiPassRunner
    from engine.tokenizer.wake_tokenizer import WakeTokenizer
    from lenses.registry import get_lens

    model = WakeModel.from_pretrained('meta-llama/Llama-3.1-8B')
    tokenizer = WakeTokenizer('engine/tokenizer/morpheme_db/seed_morphemes.json', graph_client=None)
    lenses = {name: get_lens(name) for name in LENS_NAMES}
    runner = MultiPassRunner(model, lenses, probes={}, tokenizer=tokenizer)

    results = []
    for p in FALLBACK_PASSAGES:
        wake_tokens = tokenizer.tokenize_passage(p['passage'], p['page'], p['line'])
        r = runner.run(p['passage'], wake_tokens, p['page'], p['line'])
        results.append(r)
else:
    # Synthetic residual streams for offline exploration
    np.random.seed(42)
    D = 256  # reduced dim for demo

    # Generate lens-specific cluster centres
    lens_centres = {name: np.random.randn(D) * 3 for name in LENS_NAMES}

    results = []
    for p in FALLBACK_PASSAGES:
        tokens = p['passage'].split()
        residuals_by_lens = {}
        for name, centre in lens_centres.items():
            # Mix: most tokens near centre, some between centres (superposed)
            n = len(tokens)
            noise = np.random.randn(n, D) * 0.5
            base = np.tile(centre, (n, 1))
            # ~20% of tokens are 'superposed' — placed midway between two lenses
            sup_mask = np.random.rand(n) < 0.2
            other = lens_centres[np.random.choice(LENS_NAMES)]
            base[sup_mask] = (centre + other) / 2
            residuals_by_lens[name] = base + noise

        results.append({
            'passage': p['passage'],
            'tokens': tokens,
            'residuals_by_lens': residuals_by_lens,
        })

print(f'Loaded {len(results)} passages')

In [ ]:
# ── PCA geometry per passage ────────────────────────────────────────────────
from interpretability.residual.stream_geometry import ResidualStreamAnalyzer

analyzer = ResidualStreamAnalyzer(n_components=2)
all_classifications = []  # 'superposed' | 'collapsed' | 'opaque'
geometry_results = []

for r in results:
    if USE_REAL_MODEL:
        mid_layer = list(r.pass_results[LENS_NAMES[0]].residual_streams.keys())[2]
        residuals = {name: pr.residual_streams[mid_layer]
                     for name, pr in r.pass_results.items()}
        tokens = r.pass_results[LENS_NAMES[0]].tokens
        passage = r.passage
    else:
        residuals = r['residuals_by_lens']
        tokens = r['tokens']
        passage = r['passage']

    geo = analyzer.compute_geometry(residuals, tokens, layer=16, passage=passage)
    geometry_results.append(geo)

    for pos in range(len(tokens)):
        point = geo.pca_coords[pos]  # using first lens as reference
        is_sup = pos in geo.superposition_tokens
        # Simple classification heuristic for synthetic data
        label = 'superposed' if is_sup else 'collapsed'
        all_classifications.append(label)

from collections import Counter
counts = Counter(all_classifications)
print('Classification distribution:', dict(counts))
total = sum(counts.values())
for label, n in counts.items():
    print(f'  {label}: {n} ({100*n/total:.1f}%)')

In [ ]:
# ── SuperpositionMap visualisation ──────────────────────────────────────────
# Plot PCA for the first passage with all lenses overlaid
geo = geometry_results[0]
passage_tokens = results[0]['tokens'] if not USE_REAL_MODEL else results[0].pass_results[LENS_NAMES[0]].tokens

fig = go.Figure()

SYMBOLS = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up']
n_tokens = len(passage_tokens)
n_lenses = len(LENS_NAMES)

for i, (lens_name, color) in enumerate(zip(LENS_NAMES, LENS_COLORS)):
    if USE_REAL_MODEL:
        mid_layer = list(results[0].pass_results[lens_name].residual_streams.keys())[2]
        rs = results[0].pass_results[lens_name].residual_streams[mid_layer]
        pca = PCA(n_components=2).fit_transform(rs)
    else:
        rs = results[0]['residuals_by_lens'][lens_name]
        pca = PCA(n_components=2).fit_transform(rs)

    sup_mask = [j in geo.superposition_tokens for j in range(len(passage_tokens))]

    fig.add_trace(go.Scatter(
        x=pca[:, 0], y=pca[:, 1],
        mode='markers+text',
        marker=dict(
            symbol=[SYMBOLS[i % len(SYMBOLS)]] * len(pca),
            size=[14 if s else 8 for s in sup_mask],
            color=color,
            line=dict(width=2, color='white'),
            opacity=0.85
        ),
        text=[t if s else '' for t, s in zip(passage_tokens, sup_mask)],
        textposition='top center',
        name=lens_name,
    ))

fig.update_layout(
    title=f'SuperpositionMap — Residual Stream Geometry (PCA layer 16)<br>Passage: "{results[0]["passage"][:60]}..."',
    paper_bgcolor='#0d1117',
    plot_bgcolor='#161b22',
    font=dict(color='#c9d1d9'),
    legend=dict(bgcolor='#21262d', bordercolor='#30363d'),
    xaxis=dict(title='PC1', gridcolor='#21262d'),
    yaxis=dict(title='PC2', gridcolor='#21262d'),
    width=900, height=600
)

fig.show()
fig.write_html('e2_superposition_map.html')
print('Saved e2_superposition_map.html')

In [ ]:
# ── Chi-squared test: is the distribution non-uniform? ──────────────────────
observed = np.array([counts.get(c, 0) for c in ['superposed', 'collapsed', 'opaque']])
expected = np.full_like(observed, observed.sum() / 3, dtype=float)

from scipy.stats import chisquare
stat, p = chisquare(observed, f_exp=expected)
print(f'χ² = {stat:.3f}, p = {p:.4f}')
print('Distribution is', 'NON-UNIFORM (p<0.05) ✓' if p < 0.05 else 'uniform (p≥0.05)')

sup_pct = 100 * counts.get('superposed', 0) / max(total, 1)
print(f'Superposed tokens: {sup_pct:.1f}% (target ≥10%): {"✓" if sup_pct >= 10 else "✗"}')

## Interpretation

**Superposed tokens** (residual stream between lens clusters) are the primary empirical signal
for the PKD hypothesis. If a token's residual stream falls roughly equidistant from the Viconian
and Kabbalistic centroids — rather than near one and far from the other — the model is holding
both frameworks simultaneously.

**Collapsed tokens** are tokens where the model chose one reading. This is not a failure — it
may reflect the lens context doing its job. The interesting question is whether *the same token*
collapses to *different lenses* under different prompting conditions.

**Opaque tokens** are genuine parse failures: the residual stream occupies a low-density region
far from all lens centroids. These are candidates for new lens discovery (Experiment 3).

Next: run E3 to extract the implicit query fingerprints of opaque tokens and cluster them.